# 01 — Collect Tracking and Face Observations

## Responsibility

Run the expensive video stage once: inspect a selected source range, detect and track people, collect associated face embeddings, and save a reusable observation run.

This notebook **does not resolve logical person IDs, select a target, crop frames, or render video**. Its output is the input to `02_resolve_identities.ipynb`.

## 1. Configuration

Specify a time range or an absolute frame range. When `START_FRAME` or `END_FRAME` is set, it overrides the corresponding time value.

In [ ]:
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIRECTORY.parent if WORKING_DIRECTORY.name == "notebooks" else WORKING_DIRECTORY

INPUT_VIDEO = PROJECT_ROOT / "input" / "source04_fixed.mp4"
START_SECONDS = 0.0
END_SECONDS = -1.0  # -1 means through the end of the video
START_FRAME = None
END_FRAME = None  # exclusive

RUN_NAME = None  # None creates a name from the video and resolved frame range
SAVE_FACE_CROPS = True

FACE_DETECTION_HZ = 15
MAX_FACES_PER_FRAME = 0  # 0 means unlimited
YOLO_IMAGE_SIZE = 512
YOLO_MAX_DETECTIONS = 20
YOLO_CONFIDENCE = 0.25
YOLO_MODEL_NAME = "yolo11m.pt"
TRACKER_NAME = "deepocsort.yaml"

FACE_MODEL_NAME = "buffalo_l"
FACE_DETECTION_SIZE = (640, 640)
FACE_DETECTION_THRESHOLD = 0.35

## 2. Load project code and resolve the source range

In [ ]:
import sys

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

import cv2
import onnxruntime as ort
import torch

from person_tracker.io import inspect_video
from person_tracker.video import resolve_processing_range

video_info = inspect_video(INPUT_VIDEO)
time_range = resolve_processing_range(video_info, START_SECONDS, END_SECONDS)

start_frame = time_range.start_frame if START_FRAME is None else int(START_FRAME)
end_frame = time_range.end_frame if END_FRAME is None else int(END_FRAME)

if not 0 <= start_frame < end_frame <= video_info.frame_count:
    raise ValueError(
        f"Invalid frame range {start_frame}:{end_frame}; "
        f"source contains {video_info.frame_count} frames"
    )

run_name = RUN_NAME or f"{Path(video_info.path).stem}-{start_frame:08d}-{end_frame:08d}"
RUN_DIRECTORY = PROJECT_ROOT / "runs" / run_name

print(f"Source: {video_info.path}")
print(f"Range: frames {start_frame}:{end_frame} ({start_frame / video_info.fps:.3f}s–{end_frame / video_info.fps:.3f}s)")
print(f"Run directory: {RUN_DIRECTORY}")

## 3. Load detection and recognition models

YOLO weights and InsightFace files are kept under the project `models/` directory. Download progress remains visible on the first run.

In [ ]:
import warnings

from insightface.app import FaceAnalysis
from ultralytics import YOLO

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="insightface.utils.face_align",
)

MODELS_DIRECTORY = PROJECT_ROOT / "models"
MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)
YOLO_MODEL = MODELS_DIRECTORY / YOLO_MODEL_NAME

YOLO_DEVICE = 0 if torch.cuda.is_available() else "cpu"
FACE_PROVIDERS = (
    ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if "CUDAExecutionProvider" in ort.get_available_providers()
    else ["CPUExecutionProvider"]
)

model = YOLO(str(YOLO_MODEL))
face_app = FaceAnalysis(
    name=FACE_MODEL_NAME,
    root=str(PROJECT_ROOT),  # InsightFace appends models/<bundle name>
    allowed_modules=["detection", "recognition"],
    providers=FACE_PROVIDERS,
)
face_app.prepare(
    ctx_id=0 if FACE_PROVIDERS[0] == "CUDAExecutionProvider" else -1,
    det_size=FACE_DETECTION_SIZE,
    det_thresh=FACE_DETECTION_THRESHOLD,
)

print(f"YOLO device: {YOLO_DEVICE}")
print(f"InsightFace providers: {FACE_PROVIDERS}")

## 4. Collect observations

An interrupted run saves the observations collected before the interruption, then re-raises the interruption.

In [ ]:
from tqdm.auto import tqdm

from person_tracker.face import extract_track_faces
from person_tracker.storage import build_run_manifest, save_observation_run

cap = cv2.VideoCapture(str(INPUT_VIDEO))
if not cap.isOpened():
    raise RuntimeError(f"Could not open {INPUT_VIDEO}")
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

face_detection_interval = max(1, round(video_info.fps / FACE_DETECTION_HZ))
tracking_history = {}
face_samples = []
collection_error = None

try:
    with tqdm(
        total=end_frame - start_frame,
        desc="Tracking + face analysis",
        unit="frame",
    ) as progress:
        for frame_no in range(start_frame, end_frame):
            ok, frame = cap.read()
            if not ok:
                raise RuntimeError(f"Could not decode frame {frame_no}")

            result = model.track(
                frame,
                persist=True,
                tracker=TRACKER_NAME,
                classes=[0],
                conf=YOLO_CONFIDENCE,
                imgsz=YOLO_IMAGE_SIZE,
                max_det=YOLO_MAX_DETECTIONS,
                device=YOLO_DEVICE,
                verbose=False,
            )[0]

            frame_tracks = []
            if result.boxes.id is not None:
                boxes = result.boxes.xyxy.cpu().numpy()
                ids = result.boxes.id.int().cpu().numpy()
                confidences = result.boxes.conf.cpu().numpy()

                for box, track_id, confidence in zip(boxes, ids, confidences):
                    frame_tracks.append({
                        "track_id": int(track_id),
                        "bbox": box.astype(int),
                        "confidence": float(confidence),
                    })

                if (frame_no - start_frame) % face_detection_interval == 0:
                    face_samples.extend(extract_track_faces(
                        face_app,
                        frame,
                        frame_no,
                        boxes,
                        ids,
                        max_faces=MAX_FACES_PER_FRAME,
                    ))

            tracking_history[frame_no] = frame_tracks
            progress.update(1)
except BaseException as error:
    collection_error = error
finally:
    cap.release()

    actual_end_frame = max(tracking_history, default=start_frame - 1) + 1
    manifest = build_run_manifest(
        video_path=INPUT_VIDEO,
        width=video_info.width,
        height=video_info.height,
        fps=video_info.fps,
        frame_count=video_info.frame_count,
        start_frame=start_frame,
        end_frame=actual_end_frame,
        settings={
            "requested_end_frame": end_frame,
            "complete": collection_error is None and actual_end_frame == end_frame,
            "yolo_model": YOLO_MODEL_NAME,
            "tracker": TRACKER_NAME,
            "yolo_confidence": YOLO_CONFIDENCE,
            "yolo_image_size": YOLO_IMAGE_SIZE,
            "yolo_max_detections": YOLO_MAX_DETECTIONS,
            "face_model": FACE_MODEL_NAME,
            "face_detection_hz": FACE_DETECTION_HZ,
            "face_detection_size": FACE_DETECTION_SIZE,
            "face_detection_threshold": FACE_DETECTION_THRESHOLD,
            "max_faces_per_frame": MAX_FACES_PER_FRAME,
        },
    )
    save_observation_run(
        RUN_DIRECTORY,
        manifest=manifest,
        tracking_history=tracking_history,
        face_samples=face_samples,
        save_face_crops=SAVE_FACE_CROPS,
    )

if collection_error is not None:
    raise collection_error

## 5. Saved run summary

In [ ]:
track_observations = sum(len(tracks) for tracks in tracking_history.values())

print(f"Saved run: {RUN_DIRECTORY}")
print(f"Frames processed: {len(tracking_history)}")
print(f"Track observations: {track_observations}")
print(f"Face samples: {len(face_samples)}")
print("Next: open 02_resolve_identities.ipynb and set RUN_DIRECTORY to this path.")